# DEPTH — EXP-002 training (Kaggle, GPU T4 ×2)

Trains the next DEPTH detector on dataset **v2b** and packages each model for one-command onboarding.

* **EXP-002** — YOLO11s at **1024 px** on full frames + 2×2 tiles (GPU 0)
* **EXP-002s** — the same at **640 px** (GPU 1) — STUDY-13 showed resolution must be *chosen* on
  held-out recordings, not assumed; onboarding compares the two within a speed budget.

Each run saves every epoch and exports the checkpoint with the best **ghost-gear AP@0.5 on the
held-out validation recordings** (not ultralytics' default box-tightness fitness).

**Settings:** Accelerator **GPU T4 ×2** · Internet **On** · then **Save Version → Save & Run All
(Commit)**. With one GPU the two runs go one after the other. Outputs: `EXP-002_complete.zip`,
`EXP-002s_complete.zip` in the Output tab.

In [ ]:
# 1. environment — the repo + the EXACT ultralytics version the kit was tested with
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!rm -rf depth && git clone -q https://github.com/madhesh60/depth.git && cd depth && git log --oneline -1
%cd /kaggle/working/depth
!pip install -q ultralytics==8.4.157
import torch, ultralytics, cv2
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA GPUs", torch.cuda.device_count(), "| cv2", cv2.__version__)
assert ultralytics.__version__ == "8.4.157", "wrong ultralytics version"
assert torch.cuda.is_available(), "no GPU - set Accelerator to GPU T4 x2"

In [ ]:
# 2. find the uploaded dataset (the folder holding data.yaml + train/ val/ test/) — no manual path
import glob, os
hits = [p for p in glob.glob("/kaggle/input/**/data.yaml", recursive=True) if os.path.isdir(os.path.join(os.path.dirname(p), "train"))]
assert hits, "dataset not found - add the depth-v2b dataset to this notebook (Add Input)"
SRC = os.path.dirname(sorted(hits, key=len)[0])
for s in ("train", "val", "test", "test_official398", "test_xsonar"):
    d = os.path.join(SRC, s, "images")
    print(f"{s:>17}: {len(os.listdir(d)) if os.path.isdir(d) else 'missing'} images")
print("SRC =", SRC)
assert len(os.listdir(os.path.join(SRC, "train", "images"))) > 1500, "train split looks incomplete"

In [ ]:
# 3. tiled training set in /kaggle/working (val/test stay the untouched full frames)
!python DATASET/scripts/build_tiles.py --src "{SRC}" --out /kaggle/working/v2b_tiles
!cat /kaggle/working/v2b_tiles/data.yaml
!echo "train images: $(ls /kaggle/working/v2b_tiles/train/images | wc -l)"
!df -h /kaggle/working | tail -1

In [ ]:
# 4. launch both runs (parallel on 2 GPUs, else one after the other) — logs in /kaggle/working/logs
import subprocess, time, torch
os.makedirs("/kaggle/working/logs", exist_ok=True)
DATA = "/kaggle/working/v2b_tiles/data.yaml"
RUNS = [  # name, imgsz, batch
    ("EXP-002", 1024, 8),
    ("EXP-002s", 640, 16),
]
def cmd(name, imgsz, batch, dev):
    return ["python", "src/detection/train.py", "--data", DATA, "--name", name, "--imgsz", str(imgsz),
            "--batch", str(batch), "--device", str(dev), "--workers", "2", "--cache", "none",
            "--epochs", "30", "--patience", "8", "--notes", f"Kaggle T4, {imgsz}px, tiles, ghost-AP50 selection"]
ngpu = torch.cuda.device_count()
procs = []
if ngpu >= 2:
    for dev, (name, imgsz, batch) in enumerate(RUNS):
        log = open(f"/kaggle/working/logs/{name}.log", "w")
        procs.append((name, subprocess.Popen(cmd(name, imgsz, batch, dev), stdout=log, stderr=subprocess.STDOUT)))
        print("started", name, "on GPU", dev)
else:
    print("one GPU: the runs will go one after the other (cell 5 starts the second)")
    name, imgsz, batch = RUNS[0]
    log = open(f"/kaggle/working/logs/{name}.log", "w")
    procs.append((name, subprocess.Popen(cmd(name, imgsz, batch, 0), stdout=log, stderr=subprocess.STDOUT)))
T0 = time.time()

In [ ]:
# ==== DEPTH EXP-002 - STATUS / WAIT / COLLECT ====================================================
# Safe to run at any time, any number of times. It NEVER starts or restarts a training.
# 1) shows whether each training is RUNNING / FINISHED / STOPPED (reads the real process list + files,
#    so it works even after a cell error or a kernel restart)
# 2) if any training is still running: waits, printing progress every few minutes (+ time left)
# 3) when both are done: copies the model packages to /kaggle/working and says what was exported
import os, csv, json, shutil, subprocess, time

WORK = "/kaggle/working"
RUNS_DIR, LOGS = f"{WORK}/depth/runs", f"{WORK}/logs"
NAMES = ["EXP-002", "EXP-002s"]          # 1024 px and 640 px
EPOCHS = 30
CHECK_EVERY_MIN = 5


def sh(cmd):
    return subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True).stdout.strip()


def secs(etime):                          # ps etime: [[dd-]hh:]mm:ss
    d = 0
    if "-" in etime:
        d, etime = etime.split("-")
    p = [int(x) for x in etime.split(":")]
    while len(p) < 3:
        p.insert(0, 0)
    return int(d) * 86400 + p[0] * 3600 + p[1] * 60 + p[2]


def running():
    """{run name: seconds running} for training processes that are alive right now."""
    out, alive = sh("ps -eo etime,args | grep 'src/detection/train.py' | grep -v grep"), {}
    for line in out.splitlines():
        et, args = (line.strip().split(None, 1) + [""])[:2]
        for n in NAMES:
            if f"--name {n} " in args + " ":
                alive[n] = max(alive.get(n, 0), secs(et))
    return alive


def epochs(n):
    p = f"{RUNS_DIR}/{n}/results.csv"
    if not os.path.exists(p):
        return []
    return [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(open(p))]


def log_tail(n, k=4):
    p = f"{LOGS}/{n}.log"
    if not os.path.exists(p):
        return ["(no log file)"]
    with open(p, "rb") as f:
        f.seek(0, 2); f.seek(max(0, f.tell() - 8000))
        txt = f.read().decode("utf-8", "replace")
    return [l.strip() for l in txt.replace("\r", "\n").split("\n") if l.strip()][-k:]


def status():
    alive = running()
    print(time.strftime("\n=== %H:%M:%S ==="), "| GPUs:", sh("nvidia-smi --query-gpu=index,utilization.gpu,memory.used "
                                                          "--format=csv,noheader | tr '\\n' ';'") or "n/a")
    for n in NAMES:
        rows = epochs(n)
        done = os.path.exists(f"{RUNS_DIR}/{n}_complete.zip")
        state = "FINISHED (package ready)" if done else ("RUNNING" if n in alive else "STOPPED")
        line = f"{n:>9}: {state}"
        if rows:
            last = rows[-1]
            best = max(float(r["metrics/mAP50(B)"]) for r in rows)
            line += (f" | epoch {len(rows)}/{EPOCHS} | val mAP50 {float(last['metrics/mAP50(B)']):.3f}"
                     f" (best {best:.3f}) | recall {float(last['metrics/recall(B)']):.3f}")
            if n in alive and not done:
                per = alive[n] / max(1, len(rows))
                left = max(0, EPOCHS - len(rows)) * per / 60
                line += f" | running {alive[n] / 3600:.1f} h, ~{left:.0f} min of training left + ~15 min selection"
        elif n in alive:
            line += f" | starting (running {alive[n] / 60:.0f} min, no finished epoch yet)"
        print(line)
        if state == "STOPPED":
            print("           last log lines:")
            for l in log_tail(n, 6):
                print("             ", l[:160])
    return alive


alive = status()
while alive:
    time.sleep(CHECK_EVERY_MIN * 60)
    alive = status()

print("\n=== no training process is running any more - collecting ===")
ok = 0
for n in NAMES:
    z = f"{RUNS_DIR}/{n}_complete.zip"
    if not os.path.exists(z):
        print(f"!! {n}: no package. It stopped before finishing - last log lines:")
        for l in log_tail(n, 15):
            print("   ", l[:200])
        continue
    shutil.copy(z, f"{WORK}/{n}_complete.zip")
    m = json.load(open(f"{RUNS_DIR}/{n}/model_meta.json"))
    sel = (m.get("selection") or {}).get("picked") or {}
    ub = (m.get("selection") or {}).get("ultralytics_best") or {}
    print(f"OK {n}: trained {m.get('train_minutes')} min | exported {sel.get('checkpoint')} "
          f"-> ghost_gear AP50 {sel.get('ghost_ap50')} (ultralytics' own pick: {ub.get('ghost_ap50')}) | "
          f"{m.get('cv2_dnn_verified')}")
    ok += 1
if ok == len(NAMES):
    shutil.rmtree(f"{WORK}/v2b_tiles", ignore_errors=True)
print(sh(f"ls -la {WORK}/*_complete.zip 2>/dev/null") or "no packages in /kaggle/working")
print("\nDownload the *_complete.zip files: right-hand panel -> Output (/kaggle/working) -> ... -> Download.")

## Next — on your own machine

Download `EXP-002_complete.zip` and `EXP-002s_complete.zip` from the **Output** tab into the repo root, then:

```bash
python -m src.detection.onboard_model --zip EXP-002_complete.zip  --max-ms 700
python -m src.detection.onboard_model --zip EXP-002s_complete.zip --max-ms 700
```

Keep the model with the better **recall ceiling / recall promise on the validation recordings within
the speed budget** (never pick by test), then `DEPTH_MODEL=<name>`.